In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve

Примонтировала свой диск к блокноту, чтобы брать данные напрямую из диска, не храня на своем компьютере

In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
train_path = "/content/drive/MyDrive/credit_offer_alpha/train_apps.csv"
test_path = "/content/drive/MyDrive/credit_offer_alpha/test_apps.csv"

In [6]:
train_apps = pd.read_csv(train_path)
val_apps = pd.read_csv(test_path)

In [7]:
X = train_apps.drop(columns=["target_value"])
y = train_apps["target_value"]

Обработаем отдельно даты - колонку decision_day

In [9]:
X["decision_day"] = pd.to_datetime(X["decision_day"])

In [10]:
X["Year"] = X["decision_day"].dt.year
X["Month"] = X["decision_day"].dt.month
X["Day"] = X["decision_day"].dt.day
X["DayOfWeek"] = X["decision_day"].dt.dayofweek
X["IsWeekend"] = X["DayOfWeek"].isin([5, 6]).astype(int)

In [11]:
X = X.drop(columns=["decision_day"])

Заполним числовые пропуски медианой

In [14]:
numeric_cols = X.select_dtypes(include=['number']).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

Заполним категориальные пропуски модой

In [20]:
cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
X[cat_cols] = X[cat_cols].fillna(X[cat_cols].mode().iloc[0])

CatBoost

In [21]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(auto_class_weights='Balanced', iterations=100, learning_rate=0.1, verbose=False)

scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train, cat_features=cat_cols)
    preds = model.predict(X_test)

    score = roc_auc_score(y_test, preds)
    scores.append(score)

print(f"Средняя ROC-AUC Score: {np.mean(scores):.4f}")


Средняя ROC-AUC Score: 0.7422


Получим значения на валидационной выборке val_apps

1. обработка дат

In [22]:
val_apps["decision_day"] = pd.to_datetime(val_apps["decision_day"])

In [23]:
for df in [val_apps]:
    df["Year"] = df["decision_day"].dt.year
    df["Month"] = df["decision_day"].dt.month
    df["Day"] = df["decision_day"].dt.day
    df["DayOfWeek"] = df["decision_day"].dt.dayofweek
    df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)

In [24]:
val_apps = val_apps.drop(columns=["decision_day"])

Заполним пропуски

In [25]:
# val_apps = val_apps.fillna(val_apps.median())
numeric_cols = val_apps.select_dtypes(include=['number']).columns
val_apps[numeric_cols] = val_apps[numeric_cols].fillna(val_apps[numeric_cols].median())

In [26]:
cat_cols = val_apps.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
val_apps[cat_cols] = val_apps[cat_cols].fillna(val_apps[cat_cols].mode().iloc[0])

Predict

In [27]:
# val_apps_scaled = val_apps_scaled.reindex(columns=X_test.columns, fill_value=0)
y_pred_proba = model.predict_proba(val_apps)

In [28]:
y_pred_proba[:, 1]

array([0.82088077, 0.78822881, 0.61870111, ..., 0.68011072, 0.73342733,
       0.65147142])

In [29]:
ans = pd.DataFrame({'front_id': val_apps['front_id'], 'target_value': y_pred_proba[:, 1]})

Выгружаем данные

In [30]:
from google.colab import files

In [31]:
ans.to_csv("catboost-offer.csv", index=False)

In [32]:
files.download("catboost-offer.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>